# 08. Finger Pose Regression

> Use the data-glove channels (CH63–67, one per finger) as continuous targets and decode finger flexion from ECoG, instead of treating gesture as a discrete label.

Reports per-finger Pearson correlation on held-out trials. This is the more demanding analogue of the classification task and a common evaluation in the ECoG hand-decoding literature.

In [ ]:
#| default_exp regression

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import numpy as np
from scipy.stats import pearsonr
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from br41n_ecog_hand_pose.epoching import find_cue_onsets, epoch

## Per-trial finger targets

Aggregate the 5-channel glove signal in the stimulus window into one scalar per finger per trial. Mean flexion in the cue window is the simplest, cleanest summary; it ignores temporal dynamics but is enough to test whether ECoG band power encodes the static hand pose.

In [ ]:
#| export
def glove_targets(rec, tmin=0.0, tmax=2.0):
    """Return `(targets, kept)`: mean flexion per finger per kept trial."""
    onsets, _ = find_cue_onsets(rec.labels)
    epochs, kept = epoch(rec.glove, onsets, rec.fs, tmin=tmin, tmax=tmax)
    return epochs.mean(axis=-1), kept   # (trials, 5)

## Ridge regression with per-finger correlation

L2-regularized linear regression handles the high feature/trial ratio (240 features × 90 trials) and predicts all 5 fingers jointly. Pearson `r` is the standard metric in the ECoG hand-decoding literature; MSE is reported alongside for completeness.

In [ ]:
#| export
def cross_validate_regression(X, Y, alpha=1.0, n_splits=5, random_state=0):
    """K-fold CV of Ridge regression. Returns `(corrs, mses)` shaped `(folds, n_targets)`."""
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    n_targets = Y.shape[1]
    corrs, mses = [], []
    for tr, te in kf.split(X):
        model = Pipeline([('scale', StandardScaler()), ('reg', Ridge(alpha=alpha))])
        model.fit(X[tr], Y[tr])
        pred = model.predict(X[te])
        corrs.append([pearsonr(pred[:, i], Y[te, i])[0] for i in range(n_targets)])
        mses.append(((pred - Y[te]) ** 2).mean(axis=0))
    return np.array(corrs), np.array(mses)

## Run

In [ ]:
%config InlineBackend.figure_format = 'retina'

import matplotlib.pyplot as plt
from br41n_ecog_hand_pose.data import load_ecog, FINGER_NAMES
from br41n_ecog_hand_pose.preprocessing import preprocess
from br41n_ecog_hand_pose.epoching import epoch_recording
from br41n_ecog_hand_pose.features import multi_band_power

plt.rcParams.update({
    'axes.grid':      True,
    'grid.linestyle': ':',
    'grid.linewidth': 0.5,
    'grid.alpha':     0.6,
})

In [ ]:
#| eval: false
rec = load_ecog()
clean, _ = preprocess(rec.ecog, rec.fs)
epochs, _ = epoch_recording(rec, tmin=0.0, tmax=2.0, signal=clean)
X = multi_band_power(epochs, rec.fs)
Y, _ = glove_targets(rec, tmin=0.0, tmax=2.0)
print(f'X={X.shape}, Y={Y.shape}')

In [ ]:
#| eval: false
corrs, mses = cross_validate_regression(X, Y, alpha=1.0)
print('Per-finger Pearson r (mean across folds):')
for fi, name in enumerate(FINGER_NAMES):
    print(f'  {name:>7}: {corrs[:, fi].mean():.3f} \u00b1 {corrs[:, fi].std():.3f}')

In [ ]:
#| eval: false
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(FINGER_NAMES, corrs.mean(axis=0), yerr=corrs.std(axis=0), capsize=3)
ax.axhline(0, color='k', lw=0.8)
ax.set_ylabel('Pearson r')
ax.set_title('Cross-validated finger-flexion decoding from ECoG band power')
plt.tight_layout(); plt.show()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()